# Holt's Exponential Smoothing (Double Exponential Smoothing)

Simple Exponential Smoothing (SES) only tracks the **level** of a series — so when the data **trends up or down**, SES always lags behind.

**Holt's method** fixes this by smoothing **two** things at once:

- the **level** $L_t$ — where the series *is* right now
- the **trend** $T_t$ — how fast it is *moving* per period

It therefore uses **two** smoothing parameters: $\alpha$ for the level and $\beta$ for the trend.

## 1. The three equations

**Level** (like SES, but the previous *level + trend* is the baseline):
$$\hat{L}_t = \alpha\,y_t + (1-\alpha)\,(\hat{L}_{t-1} + \hat{T}_{t-1})$$

**Trend** (smooth the change in level):
$$\hat{T}_t = \beta\,(\hat{L}_t - \hat{L}_{t-1}) + (1-\beta)\,\hat{T}_{t-1}$$

**Forecast** (level plus $h$ steps of trend):
$$\hat{y}_{t+h\mid t} = \hat{L}_t + h\,\hat{T}_t$$

For a **one-step** forecast $h=1$: $\ \hat{y}_{t+1} = \hat{L}_t + \hat{T}_t$.

**Initialisation** (common choice):
$$\hat{L}_1 = y_1, \qquad \hat{T}_1 = 0$$

## 2. By-hand worked example

Data (monthly), with $\alpha = 0.2,\ \beta = 0.2$:

| Month | Feb(1) | Mar(2) | Apr(3) | May(4) | Jun(5) | Jul(6) | Aug(7) | Sep(8) | Oct(9) | Nov(10) |
|---|---|---|---|---|---|---|---|---|---|---|
| **Actual** | 26 | 8 | 17 | 29 | 34 | 17 | 22 | 19 | 16 | 22 |

We **train** on Feb–Aug (1–7) and **test** on Sep–Nov (8–10).

### Step 0 — initialise
$$\hat{L}_1 = y_1 = 26, \qquad \hat{T}_1 = 0$$
$$\hat{y}_2 = \hat{L}_1 + \hat{T}_1 = 26 + 0 = 26$$

### Step 1 — update at month 2 (Mar, $y_2 = 8$)
$$\hat{L}_2 = \alpha y_2 + (1-\alpha)(\hat{L}_1+\hat{T}_1) = 0.2(8) + 0.8(26+0) = \mathbf{22.40}$$
$$\hat{T}_2 = \beta(\hat{L}_2-\hat{L}_1) + (1-\beta)\hat{T}_1 = 0.2(22.40-26) + 0.8(0) = \mathbf{-0.72}$$
$$\hat{y}_3 = \hat{L}_2 + \hat{T}_2 = 22.40 - 0.72 = \mathbf{21.68}$$

### Step 2 — update at month 3 (Apr, $y_3 = 17$)
$$\hat{L}_3 = 0.2(17) + 0.8(22.40 - 0.72) = \mathbf{20.74}$$
$$\hat{T}_3 = 0.2(20.74 - 22.40) + 0.8(-0.72) = \mathbf{-0.91}$$
$$\hat{y}_4 = \hat{L}_3 + \hat{T}_3 = 20.74 - 0.91 = \mathbf{19.84}$$

…and so on, rolling forward one month at a time. Notice the trend $\hat{T}$ is **negative**, so each forecast is nudged downward — Holt has detected the (mild) downward drift.

## 3. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)

months = ["Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]
actual = [26, 8, 17, 29, 34, 17, 22, 19, 16, 22]
df = pd.DataFrame({"month": months, "actual": actual}, index=range(1, 11))
df.index.name = "t"
df

## 4. Reproduce the by-hand table in code

We loop the two update equations over the **training** months (t = 2 … 7), recording $\hat{L}_t$, $\hat{T}_t$ and the one-step forecast $\hat{y}_t$.

In [ ]:
alpha, beta = 0.2, 0.2
y = df["actual"].to_dict()          # {1:26, 2:8, ...}
train_end = 7                        # Aug is the last training month

L = {1: float(y[1])}                 # L_1 = y_1 = 26
T = {1: 0.0}                         # T_1 = 0
yhat = {}

for t in range(2, train_end + 1):
    yhat[t] = L[t-1] + T[t-1]                                   # forecast made last step
    L[t]    = alpha * y[t] + (1 - alpha) * (L[t-1] + T[t-1])    # level update
    T[t]    = beta  * (L[t] - L[t-1]) + (1 - beta) * T[t-1]     # trend update

table = pd.DataFrame({
    "actual":  [y[t] for t in range(1, train_end + 1)],
    "L":       [L[t] for t in range(1, train_end + 1)],
    "T":       [T[t] for t in range(1, train_end + 1)],
    "forecast":[np.nan] + [yhat[t] for t in range(2, train_end + 1)],
}, index=range(1, train_end + 1)).round(2)
table

The `forecast` column reads **26, 21.68, 19.84, …** — exactly the predictions from the slide. ✅

## 5. Forecast the test months (multi-step)

After training, the level and trend are **frozen** at their last values $\hat{L}_7,\ \hat{T}_7$. We forecast Sep/Oct/Nov as $h = 1, 2, 3$ steps ahead from Aug:

$$\hat{y}_{7+h} = \hat{L}_7 + h\,\hat{T}_7$$

In [ ]:
test_preds = {7 + h: L[train_end] + h * T[train_end] for h in (1, 2, 3)}

test_df = pd.DataFrame({
    "month":   [months[t-1] for t in test_preds],
    "actual":  [y[t] for t in test_preds],
    "forecast":[round(v, 2) for v in test_preds.values()],
}, index=list(test_preds))
test_df

Predictions **21.75, 21.45, 21.15** — matching the slide (21.75, 21.46, 21.17 up to rounding). Each step subtracts the small negative trend, so the forecast slowly drifts down.

## 6. Evaluate on the test set — RMSE & MAPE

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}, \qquad \text{MAPE} = \frac{100}{n}\sum\left|1 - \frac{\hat{y}_i}{y_i}\right|$$

In [ ]:
act  = test_df["actual"].to_numpy(dtype=float)
pred = np.array(list(test_preds.values()))
err  = act - pred
n    = len(act)

sse  = np.sum(err ** 2)
mse  = sse / n
rmse = np.sqrt(mse)
mape = np.mean(np.abs(err / act)) * 100

print(f"sum of squared errors = {sse:.2f}")
print(f"MSE  (SSE / n)        = {mse:.2f}")
print(f"RMSE                  = {rmse:.2f}")
print(f"MAPE                  = {mape:.2f}%")

> **⚠️ Note on the slide's RMSE.** The slide writes
> $$\text{RMSE} = \sqrt{\tfrac{(19-21.75)^2 + (16-21.46)^2 + (22-21.17)^2}{3}} = \sqrt{38.06} = 6.17$$
> The number **38.06 is the *sum* of the squared errors, not the sum divided by 3** — so $\sqrt{38.06}=6.17$ actually skips the `/3` step. The correct RMSE is
> $$\sqrt{38.06 / 3} = \sqrt{12.69} \approx \mathbf{3.56}.$$
> The **MAPE** on the slide (17.46%) is computed correctly. Our code above gives the right RMSE.

## 7. Plot actual vs forecast

In [ ]:
all_fc = [np.nan] + [yhat[t] for t in range(2, train_end + 1)] + list(test_preds.values())

plt.plot(range(1, 11), actual, "ko-", label="actual")
plt.plot(range(1, 8), all_fc[:7], "b.--", label="fitted (train)")
plt.plot(range(8, 11), all_fc[7:], "rs--", label="forecast (test)")
plt.axvline(7.5, color="grey", ls=":")
plt.xticks(range(1, 11), months)
plt.ylabel("value"); plt.title("Holt's method: α=0.2, β=0.2")
plt.legend(); plt.show()

## 8. The same thing with statsmodels

In practice you use `statsmodels` `Holt`, which can also **optimise** $\alpha$ and $\beta$ for you.

In [ ]:
from statsmodels.tsa.holtwinters import Holt

train = np.array(actual[:7], dtype=float)

# Fixed alpha=beta=0.2 to match the by-hand example
fit = Holt(train, initialization_method="known",
           initial_level=train[0], initial_trend=0.0
          ).fit(smoothing_level=0.2, smoothing_trend=0.2, optimized=False)

print("3-step forecast:", np.round(fit.forecast(3), 2))   # ~ [21.74, 21.45, 21.15]

## 9. Summary

- **Holt = SES + a trend term.** Two equations (level & trend), two parameters ($\alpha,\ \beta$).
- Forecast = $\hat{L}_t + h\,\hat{T}_t$, so it **projects the trend** $h$ steps ahead (unlike SES, which forecasts a flat line).
- $\alpha$ controls how fast the **level** adapts; $\beta$ how fast the **trend** adapts.
- Handles data with a **trend** but **no seasonality**. Add a seasonal term → **Holt-Winters** (triple exponential smoothing).
- Evaluate forecasts on a held-out test set with **RMSE** (penalises big misses, in original units) and **MAPE** (scale-free %).